In [1]:
# ==========================================================
# INSTALL REQUIRED LIBRARIES
# ==========================================================

!pip -q install requests joblib geopy

In [2]:
# ==========================================================
# IMPORT LIBRARIES
# ==========================================================

import os
import joblib
import requests
import warnings
import numpy as np
import pandas as pd

from datetime import datetime

import torch
import torch.nn as nn

from google.colab import drive

warnings.filterwarnings("ignore")

In [3]:
# ==========================================================
# MOUNT GOOGLE DRIVE
# ==========================================================

drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import os
import joblib

PROJECT_DIR = "/content/drive/MyDrive/AQI_BiLSTM_Project"

print(os.path.exists(PROJECT_DIR))

print(os.listdir(PROJECT_DIR))

feature_scaler = joblib.load(
    os.path.join(PROJECT_DIR, "feature_scaler.pkl")
)

print("Loaded successfully")

True
['best_bilstm_model.pth', 'feature_scaler.pkl', 'target_scaler.pkl', 'city_encoder.pkl', 'state_encoder.pkl', 'feature_columns.pkl', 'ML_Ready_Air_Quality_Dataset.csv']
Loaded successfully


In [8]:
print(PROJECT_DIR)
print(os.listdir(PROJECT_DIR))

/content/drive/MyDrive/AQI_BiLSTM_Project
['best_bilstm_model.pth', 'feature_scaler.pkl', 'target_scaler.pkl', 'city_encoder.pkl', 'state_encoder.pkl', 'feature_columns.pkl', 'ML_Ready_Air_Quality_Dataset.csv']


In [11]:
PROJECT_DIR = "/content/drive/MyDrive/AQI_BiLSTM_Project"

DATASET_PATH = os.path.join(
    PROJECT_DIR,
    "ML_Ready_Air_Quality_Dataset.csv"
)

In [12]:
# ==========================================================
# LOAD DATASET
# ==========================================================

df = pd.read_csv(DATASET_PATH)

print(df.shape)

df.head()

(3431900, 25)


,Timestamp,PM2.5,PM10,Nitric Oxide,Nitrogen Dioxide,Nitrogen Oxides,Ammonia,Sulfur Dioxide,Carbon Monoxide,Ozone,...,City,Latitude,Longitude,Calculated_AQI,Month,Hour,DayOfWeek,Is_Weekend,State_Encoded,City_Encoded
0,2017-09-05 11:00:00,25.0,45.0,1.80,12.20,7.90,10.20,5.60,0.10,79.5,...,"Anand Kala Kshetram, Rajamahendravaram (, )",16.987287,81.736318,90.105556,9,11,1,0,0,3
1,2017-09-05 12:00:00,25.0,45.0,1.80,12.20,7.90,10.20,5.60,0.10,79.5,...,"Anand Kala Kshetram, Rajamahendravaram (, )",16.987287,81.736318,90.105556,9,12,1,0,0,3
2,2017-09-05 13:00:00,25.0,45.0,1.80,12.20,7.90,10.20,5.60,0.10,79.5,...,"Anand Kala Kshetram, Rajamahendravaram (, )",16.987287,81.736318,90.105556,9,13,1,0,0,3
3,2017-09-05 14:00:00,25.0,45.0,1.80,12.20,7.90,10.20,5.60,0.10,79.5,...,"Anand Kala Kshetram, Rajamahendravaram (, )",16.987287,81.736318,90.105556,9,14,1,0,0,3
4,2017-09-05 15:00:00,23.0,49.5,0.65,14.55,8.28,8.85,4.52,0.15,62.5,...,"Anand Kala Kshetram, Rajamahendravaram (, )",16.987287,81.736318,90.105556,9,15,1,0,0,3


In [13]:
import joblib
import os

# ==========================================
# LOAD TRAINING ARTIFACTS
# ==========================================

feature_scaler = joblib.load(
    os.path.join(PROJECT_DIR, "feature_scaler.pkl")
)

target_scaler = joblib.load(
    os.path.join(PROJECT_DIR, "target_scaler.pkl")
)

state_encoder = joblib.load(
    os.path.join(PROJECT_DIR, "state_encoder.pkl")
)

city_encoder = joblib.load(
    os.path.join(PROJECT_DIR, "city_encoder.pkl")
)

FEATURE_COLUMNS = joblib.load(
    os.path.join(PROJECT_DIR, "feature_columns.pkl")
)

print("✓ Feature Scaler Loaded")
print("✓ Target Scaler Loaded")
print("✓ State Encoder Loaded")
print("✓ City Encoder Loaded")
print("✓ Feature Columns Loaded")

print("\nTotal Features :", len(FEATURE_COLUMNS))
print(FEATURE_COLUMNS)

✓ Feature Scaler Loaded
✓ Target Scaler Loaded
✓ State Encoder Loaded
✓ City Encoder Loaded
✓ Feature Columns Loaded

Total Features : 21
['PM2.5', 'PM10', 'Nitric Oxide', 'Nitrogen Dioxide', 'Nitrogen Oxides', 'Ammonia', 'Sulfur Dioxide', 'Carbon Monoxide', 'Ozone', 'Ambient Temperature', 'Relative Humidity', 'Solar Radiation', 'Rainfall', 'Latitude', 'Longitude', 'Month', 'Hour', 'DayOfWeek', 'Is_Weekend', 'State_Encoded', 'City_Encoded']


In [14]:
import torch
import torch.nn as nn

# ==========================================
# DEVICE
# ==========================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)


# ==========================================
# BiLSTM Model
# ==========================================

class BiLSTMModel(nn.Module):

    def __init__(self, input_size):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )

        self.fc1 = nn.Linear(128,64)
        self.fc2 = nn.Linear(64,32)
        self.fc3 = nn.Linear(32,1)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

    def forward(self,x):

        output,_ = self.lstm(x)

        output = output[:,-1,:]

        output = self.dropout(output)

        output = self.relu(self.fc1(output))

        output = self.dropout(output)

        output = self.relu(self.fc2(output))

        output = self.fc3(output)

        return output

cuda


In [17]:
print(MODEL_PATH)
print(os.listdir(PROJECT_DIR))

/content/drive/MyDrive/AQI_Project/best_bilstm_model.pth
['best_bilstm_model.pth', 'feature_scaler.pkl', 'target_scaler.pkl', 'city_encoder.pkl', 'state_encoder.pkl', 'feature_columns.pkl', 'ML_Ready_Air_Quality_Dataset.csv']


In [18]:
import os

PROJECT_DIR = "/content/drive/MyDrive/AQI_BiLSTM_Project"

MODEL_PATH = os.path.join(
    PROJECT_DIR,
    "best_bilstm_model.pth"
)

print(PROJECT_DIR)
print(MODEL_PATH)
print(os.path.exists(MODEL_PATH))

/content/drive/MyDrive/AQI_BiLSTM_Project
/content/drive/MyDrive/AQI_BiLSTM_Project/best_bilstm_model.pth
True


In [20]:
# ======================================================
# ATTENTION LAYER
# ======================================================

class Attention(nn.Module):

    def __init__(self, hidden_size):
        super().__init__()

        self.attention = nn.Linear(hidden_size * 2, 1)

    def forward(self, lstm_output):

        scores = self.attention(lstm_output)

        weights = torch.softmax(scores, dim=1)

        context = torch.sum(
            weights * lstm_output,
            dim=1
        )

        return context


# ======================================================
# ATTENTION BiLSTM MODEL
# ======================================================

class AttentionBiLSTM(nn.Module):

    def __init__(self, input_size):

        super().__init__()

        self.hidden_size = 128

        self.bilstm = nn.LSTM(
            input_size=input_size,
            hidden_size=self.hidden_size,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.30
        )

        self.attention = Attention(self.hidden_size)

        self.dropout = nn.Dropout(0.30)

        self.fc1 = nn.Linear(256,128)

        self.fc2 = nn.Linear(128,64)

        self.fc3 = nn.Linear(64,1)

        self.relu = nn.ReLU()

    def forward(self,x):

        output,_ = self.bilstm(x)

        context = self.attention(output)

        x = self.dropout(context)

        x = self.relu(self.fc1(x))

        x = self.dropout(x)

        x = self.relu(self.fc2(x))

        x = self.fc3(x)

        return x

In [21]:
# ======================================================
# LOAD TRAINED MODEL
# ======================================================

model = AttentionBiLSTM(
    input_size=len(FEATURE_COLUMNS)
).to(device)

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device
)

model.load_state_dict(checkpoint)

model.eval()

print("✅ Attention BiLSTM Model Loaded Successfully")

✅ Attention BiLSTM Model Loaded Successfully


In [22]:
# ======================================================
# GET LATEST HISTORICAL FEATURES
# ======================================================

def get_latest_station_features(station_name):

    station_data = df[df["City"] == station_name].copy()

    if len(station_data) == 0:
        raise ValueError(f"{station_name} not found in dataset.")

    station_data = station_data.sort_values("Timestamp")

    latest = station_data.iloc[-1]

    return latest

In [27]:
station = "Anand Vihar ( , )"

historical = get_latest_station_features(station)

historical

,834307
Timestamp,2026-06-30 23:00:00
PM2.5,72.0
PM10,215.0
Nitric Oxide,10.63
Nitrogen Dioxide,44.63
Nitrogen Oxides,32.4
Ammonia,24.4
Sulfur Dioxide,6.43
Carbon Monoxide,2.24
Ozone,5.47


In [28]:
# ======================================================
# CREATE LIVE FEATURE VECTOR
# ======================================================

from datetime import datetime

def create_live_features(
    station_name,
    live_pollutants,
    live_weather
):

    # Get latest historical record
    row = get_latest_station_features(station_name).copy()

    # ----------------------------
    # Update pollutant values
    # ----------------------------

    row["PM2.5"] = live_pollutants["PM2.5"]
    row["PM10"] = live_pollutants["PM10"]
    row["Nitric Oxide"] = live_pollutants["Nitric Oxide"]
    row["Nitrogen Dioxide"] = live_pollutants["Nitrogen Dioxide"]
    row["Nitrogen Oxides"] = live_pollutants["Nitrogen Oxides"]
    row["Ammonia"] = live_pollutants["Ammonia"]
    row["Sulfur Dioxide"] = live_pollutants["Sulfur Dioxide"]
    row["Carbon Monoxide"] = live_pollutants["Carbon Monoxide"]
    row["Ozone"] = live_pollutants["Ozone"]

    # ----------------------------
    # Update weather values
    # ----------------------------

    row["Ambient Temperature"] = live_weather["Ambient Temperature"]
    row["Relative Humidity"] = live_weather["Relative Humidity"]
    row["Rainfall"] = live_weather["Rainfall"]
    row["Solar Radiation"] = live_weather["Solar Radiation"]

    # ----------------------------
    # Update time features
    # ----------------------------

    now = datetime.now()

    row["Month"] = now.month
    row["Hour"] = now.hour
    row["DayOfWeek"] = now.weekday()

    row["Is_Weekend"] = 1 if now.weekday() >= 5 else 0

    # ----------------------------
    # Encode State & Station
    # ----------------------------

    row["State_Encoded"] = state_encoder.transform([row["State"]])[0]

    row["City_Encoded"] = city_encoder.transform([station_name])[0]

    # ----------------------------
    # Return only training features
    # ----------------------------

    return row[FEATURE_COLUMNS]

In [29]:
station = "Anand Vihar ( , )"

live_pollutants = {

    "PM2.5":45,
    "PM10":88,
    "Nitric Oxide":12,
    "Nitrogen Dioxide":26,
    "Nitrogen Oxides":30,
    "Ammonia":14,
    "Sulfur Dioxide":8,
    "Carbon Monoxide":0.7,
    "Ozone":18

}

live_weather = {

    "Ambient Temperature":31,
    "Relative Humidity":62,
    "Rainfall":0,
    "Solar Radiation":420

}

live_features = create_live_features(
    station,
    live_pollutants,
    live_weather
)

print(live_features)

PM2.5                       45
PM10                        88
Nitric Oxide                12
Nitrogen Dioxide            26
Nitrogen Oxides             30
Ammonia                     14
Sulfur Dioxide               8
Carbon Monoxide            0.7
Ozone                       18
Ambient Temperature         31
Relative Humidity           62
Solar Radiation            420
Rainfall                     0
Latitude               28.6508
Longitude              77.3152
Month                        7
Hour                        13
DayOfWeek                    5
Is_Weekend                   1
State_Encoded                6
City_Encoded                 4
Name: 834307, dtype: object


In [30]:
# ======================================================
# SCALE FEATURES & PREDICT AQI
# ======================================================

import numpy as np

# Convert to DataFrame (preserve feature names)
live_df = pd.DataFrame([live_features])

# Scale features using training scaler
scaled_features = feature_scaler.transform(live_df)

# Convert to Tensor
input_tensor = torch.tensor(
    scaled_features,
    dtype=torch.float32
).unsqueeze(1).to(device)   # Shape -> (1,1,21)

# Prediction
with torch.no_grad():

    scaled_prediction = model(input_tensor)

# Convert back to original AQI scale
predicted_aqi = target_scaler.inverse_transform(
    scaled_prediction.cpu().numpy()
)

predicted_aqi = float(predicted_aqi[0][0])

print("=" * 50)
print(f"Predicted AQI : {predicted_aqi:.2f}")
print("=" * 50)

Predicted AQI : 162.63


In [31]:
# ======================================================
# AQI CATEGORY
# ======================================================

def get_aqi_category(aqi):

    if aqi <= 50:
        return "Good 🟢"

    elif aqi <= 100:
        return "Satisfactory 🟡"

    elif aqi <= 200:
        return "Moderate 🟠"

    elif aqi <= 300:
        return "Poor 🔴"

    elif aqi <= 400:
        return "Very Poor 🟣"

    else:
        return "Severe ⚫"


category = get_aqi_category(predicted_aqi)

print(f"AQI Category : {category}")

AQI Category : Moderate 🟠


In [35]:
# ======================================================
# FINAL RESULT
# ======================================================

print("=" * 60)
print("LIVE AQI PREDICTION")
print("=" * 60)

print(f"City         :  Delhi")
print(f"Station      : {station}")
print(f"Predicted AQI: {predicted_aqi:.2f}")
print(f"Category     : {category}")

print("=" * 60)

LIVE AQI PREDICTION
City         :  Delhi
Station      : Anand Vihar ( , )
Predicted AQI: 162.63
Category     : Moderate 🟠


In [36]:
# ======================================================
# API KEYS
# ======================================================

# Geoapify API Key
GEOAPIFY_API_KEY = "3e0e7b99b52a4930ba7e38cc481349ea"

# OpenWeather API Key
OPENWEATHER_API_KEY = "3fe20d7fd1818ba56da0a0d2239502e8"

In [37]:
# ======================================================
# IMPORTS
# ======================================================

import requests
import pandas as pd
from datetime import datetime

In [38]:
# ======================================================
# GET LATITUDE & LONGITUDE
# ======================================================

def get_coordinates(place_name):

    url = (
        "https://api.geoapify.com/v1/geocode/search"
        f"?text={place_name}"
        f"&limit=1"
        f"&apiKey={GEOAPIFY_API_KEY}"
    )

    response = requests.get(url)

    data = response.json()

    if len(data["features"]) == 0:
        raise Exception("Location not found.")

    latitude = data["features"][0]["properties"]["lat"]
    longitude = data["features"][0]["properties"]["lon"]

    return latitude, longitude

In [39]:
station = "Anand Vihar ( , )"

lat, lon = get_coordinates(station)

print("Latitude :", lat)
print("Longitude:", lon)

Latitude : 28.641115
Longitude: 77.3125024


In [40]:
# ======================================================
# GET LIVE WEATHER
# ======================================================

def get_live_weather(lat, lon):

    url = (
        "https://api.openweathermap.org/data/2.5/weather"
        f"?lat={lat}"
        f"&lon={lon}"
        f"&appid={OPENWEATHER_API_KEY}"
        "&units=metric"
    )

    response = requests.get(url)

    weather = response.json()

    result = {

        "Ambient Temperature": weather["main"]["temp"],

        "Relative Humidity": weather["main"]["humidity"],

        "Rainfall": weather.get("rain", {}).get("1h", 0),

        # OpenWeather doesn't provide solar radiation
        "Solar Radiation": historical["Solar Radiation"]

    }

    return result

In [41]:
weather = get_live_weather(lat, lon)

print(weather)

{'Ambient Temperature': 39.76, 'Relative Humidity': 29, 'Rainfall': 0, 'Solar Radiation': np.float64(15.0)}


In [42]:
# ======================================================
# GET LAST 12 RECORDS OF A STATION
# ======================================================

def get_station_sequence(station_name, sequence_length=12):

    station_df = df[df["City"] == station_name].copy()

    if len(station_df) < sequence_length:
        raise ValueError(
            f"Station '{station_name}' has fewer than {sequence_length} records."
        )

    station_df["Timestamp"] = pd.to_datetime(station_df["Timestamp"])

    station_df = station_df.sort_values("Timestamp")

    return station_df.tail(sequence_length).copy()

In [43]:
# ======================================================
# UPDATE LAST ROW WITH LIVE VALUES
# ======================================================

def update_last_row(sequence_df, live_pollutants, live_weather):

    sequence_df = sequence_df.copy()

    last_index = sequence_df.index[-1]

    # ---------------- Pollutants ----------------

    sequence_df.loc[last_index, "PM2.5"] = live_pollutants["PM2.5"]
    sequence_df.loc[last_index, "PM10"] = live_pollutants["PM10"]

    sequence_df.loc[last_index, "Nitric Oxide"] = live_pollutants["Nitric Oxide"]
    sequence_df.loc[last_index, "Nitrogen Dioxide"] = live_pollutants["Nitrogen Dioxide"]
    sequence_df.loc[last_index, "Nitrogen Oxides"] = live_pollutants["Nitrogen Oxides"]

    sequence_df.loc[last_index, "Ammonia"] = live_pollutants["Ammonia"]
    sequence_df.loc[last_index, "Sulfur Dioxide"] = live_pollutants["Sulfur Dioxide"]
    sequence_df.loc[last_index, "Carbon Monoxide"] = live_pollutants["Carbon Monoxide"]
    sequence_df.loc[last_index, "Ozone"] = live_pollutants["Ozone"]

    # ---------------- Weather ----------------

    sequence_df.loc[last_index, "Ambient Temperature"] = live_weather["Ambient Temperature"]
    sequence_df.loc[last_index, "Relative Humidity"] = live_weather["Relative Humidity"]
    sequence_df.loc[last_index, "Rainfall"] = live_weather["Rainfall"]
    sequence_df.loc[last_index, "Solar Radiation"] = live_weather["Solar Radiation"]

    # ---------------- Time ----------------

    now = datetime.now()

    sequence_df.loc[last_index, "Month"] = now.month
    sequence_df.loc[last_index, "Hour"] = now.hour
    sequence_df.loc[last_index, "DayOfWeek"] = now.weekday()
    sequence_df.loc[last_index, "Is_Weekend"] = int(now.weekday() >= 5)

    # ---------------- Encoded Columns ----------------

    sequence_df.loc[last_index, "State_Encoded"] = state_encoder.transform(
        [sequence_df.loc[last_index, "State"]]
    )[0]

    sequence_df.loc[last_index, "City_Encoded"] = city_encoder.transform(
        [sequence_df.loc[last_index, "City"]]
    )[0]

    return sequence_df

In [44]:
# ======================================================
# PREPARE MODEL INPUT
# ======================================================

def prepare_model_input(sequence_df):

    features = sequence_df[FEATURE_COLUMNS].copy()

    scaled_features = feature_scaler.transform(features)

    model_input = torch.tensor(
        scaled_features,
        dtype=torch.float32
    ).unsqueeze(0).to(device)

    return model_input

In [45]:
# ======================================================
# PREDICT AQI
# ======================================================

def predict_aqi(model_input):

    model.eval()

    with torch.no_grad():

        prediction = model(model_input)

    prediction = target_scaler.inverse_transform(
        prediction.cpu().numpy()
    )

    return float(prediction[0][0])

In [47]:
# ======================================================
# COMPLETE AQI PIPELINE
# ======================================================

station = "Anand Vihar ( , )"

# Step 1
sequence = get_station_sequence(station)

# Step 2
sequence = update_last_row(
    sequence,
    live_pollutants,
    live_weather
)

# Step 3
model_input = prepare_model_input(sequence)

print("Model Input Shape:", model_input.shape)

# Step 4
predicted_aqi = predict_aqi(model_input)

print("=" * 50)
print(f"Predicted AQI : {predicted_aqi:.2f}")
print("=" * 50)

Model Input Shape: torch.Size([1, 12, 21])
Predicted AQI : 259.81


In [51]:
for col in df.columns:
    print(col)

Timestamp
PM2.5
PM10
Nitric Oxide
Nitrogen Dioxide
Nitrogen Oxides
Ammonia
Sulfur Dioxide
Carbon Monoxide
Ozone
Ambient Temperature
Relative Humidity
Solar Radiation
Rainfall
State
City
Latitude
Longitude
Calculated_AQI
Month
Hour
DayOfWeek
Is_Weekend
State_Encoded
City_Encoded


In [52]:
station = "Anand Vihar ( , )"

last12 = get_station_sequence(station)

print(last12[["Timestamp", "Calculated_AQI"]])

                 Timestamp  Calculated_AQI
834296 2026-06-30 12:00:00      250.020833
834297 2026-06-30 13:00:00      259.875000
834298 2026-06-30 14:00:00      268.083333
834299 2026-06-30 15:00:00      274.500000
834300 2026-06-30 16:00:00      278.729167
834301 2026-06-30 17:00:00      278.545000
834302 2026-06-30 18:00:00      274.930417
834303 2026-06-30 19:00:00      273.128333
834304 2026-06-30 20:00:00      274.263750
834305 2026-06-30 21:00:00      277.097083
834306 2026-06-30 22:00:00      276.326250
834307 2026-06-30 23:00:00      275.534583


In [55]:
print(live_pollutants)

{'PM2.5': 45, 'PM10': 88, 'Nitric Oxide': 12, 'Nitrogen Dioxide': 26, 'Nitrogen Oxides': 30, 'Ammonia': 14, 'Sulfur Dioxide': 8, 'Carbon Monoxide': 0.7, 'Ozone': 18}


In [56]:
print(live_weather)

{'Ambient Temperature': 31, 'Relative Humidity': 62, 'Rainfall': 0, 'Solar Radiation': 420}


In [57]:
import requests

In [61]:
# ===========================================
# USER INPUT
# ===========================================

SOURCE = "Delhi"

DESTINATION = "Chandigarh"

# Hardcoded AQI values
SOURCE_AQI = 142
DESTINATION_AQI = 97

GEOAPIFY_API_KEY = "3e0e7b99b52a4930ba7e38cc481349ea"

In [67]:
# =====================================================
# USER INPUT
# =====================================================

print("="*60)
print("SMART AQI ROUTE RECOMMENDATION")
print("="*60)

SOURCE = input("Enter Source Location : ")

DESTINATION = input("Enter Destination Location : ")

print()

print("Source      :", SOURCE)
print("Destination :", DESTINATION)

SMART AQI ROUTE RECOMMENDATION
Enter Source Location : Delhi
Enter Destination Location : Chandigarh

Source      : Delhi
Destination : Chandigarh


In [62]:
# ===========================================
# GET LATITUDE & LONGITUDE
# ===========================================

def get_coordinates(place):

    url = (
        "https://api.geoapify.com/v1/geocode/search?"
        f"text={place}"
        f"&apiKey={GEOAPIFY_API_KEY}"
    )

    response = requests.get(url).json()

    feature = response["features"][0]

    lon, lat = feature["geometry"]["coordinates"]

    return lat, lon

In [63]:
# ===========================================
# ROUTE DISTANCE
# ===========================================

def get_route(source, destination):

    source_lat, source_lon = get_coordinates(source)

    dest_lat, dest_lon = get_coordinates(destination)

    url = (
        "https://api.geoapify.com/v1/routing?"
        f"waypoints={source_lat},{source_lon}|{dest_lat},{dest_lon}"
        "&mode=drive"
        "&details=route_details"
        f"&apiKey={GEOAPIFY_API_KEY}"
    )

    data = requests.get(url).json()

    route = data["features"][0]["properties"]

    distance = route["distance"] / 1000

    duration = route["time"] / 60

    return distance, duration

In [64]:
# ===========================================
# AQI CATEGORY
# ===========================================

def get_category(aqi):

    if aqi <= 50:
        return "Good"

    elif aqi <= 100:
        return "Satisfactory"

    elif aqi <= 200:
        return "Moderate"

    elif aqi <= 300:
        return "Poor"

    elif aqi <= 400:
        return "Very Poor"

    else:
        return "Severe"

In [65]:
# ===========================================
# HEALTH RECOMMENDATION
# ===========================================

def get_recommendation(avg_aqi):

    if avg_aqi <= 50:
        return "Safe to travel."

    elif avg_aqi <= 100:
        return "Travel is safe."

    elif avg_aqi <= 200:
        return "Sensitive people should wear a mask."

    elif avg_aqi <= 300:
        return "Wear an N95 mask while travelling."

    elif avg_aqi <= 400:
        return "Avoid unnecessary outdoor exposure."

    else:
        return "Travel is not recommended."

In [66]:
# ===========================================
# FINAL RESULT
# ===========================================

distance, duration = get_route(
    SOURCE,
    DESTINATION
)

average_aqi = (SOURCE_AQI + DESTINATION_AQI) / 2

print("="*60)

print("SMART AQI ROUTE RECOMMENDATION")

print("="*60)

print(f"Source       : {SOURCE}")
print(f"Destination  : {DESTINATION}")

print()

print(f"Source AQI       : {SOURCE_AQI}")
print(f"Destination AQI  : {DESTINATION_AQI}")
print(f"Average AQI      : {average_aqi:.2f}")

print(f"AQI Category     : {get_category(average_aqi)}")

print()

print(f"Distance         : {distance:.2f} km")
print(f"Travel Time      : {duration:.2f} minutes")

print()

print("Health Advice")

print(get_recommendation(average_aqi))

print("="*60)

SMART AQI ROUTE RECOMMENDATION
Source       : Delhi
Destination  : Chandigarh

Source AQI       : 142
Destination AQI  : 97
Average AQI      : 119.50
AQI Category     : Moderate

Distance         : 323.66 km
Travel Time      : 211.27 minutes

Health Advice
Sensitive people should wear a mask.
